# Análisis Inteligente del Consumo Energético
### Hackathon ONE · Alura + Oracle · Equipo G9

Motor de residuo sobre contrato ampliado, con el análisis exploratorio completo del notebook original del equipo.

**Definición de eficiencia:** en lugar de construir una etiqueta con una fórmula sobre las mismas entradas del modelo (circular), se predice **cuánto debería consumir** un hogar según sus características y se mide la desviación real. Es el enfoque de *benchmarking* energético de sistemas como ENERGY STAR.

> *"Esperábamos 439 kWh para un hogar de 4 personas y 120 m². Consumiste 420 — estás un 4% por debajo."*

**Contrato ampliado:**
```json
{
  "consumo_kwh": 420, "personas": 4, "superficie_m2": 120,
  "cantidad_equipos": 10, "tipo_inmueble": "Casa",
  "uso_horario_pico": true, "horas_alto_consumo": 8, "panel_solar": false
}
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

sns.set_theme(style="whitegrid")
PALETA = "Blues_r"
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
for carpeta in ["models", "outputs"]:
    Path(carpeta).mkdir(exist_ok=True)
print("Entorno listo.")

## 1. Carga del dataset

> **En Colab:** sube `consumo_energia_sudamerica.csv` antes de ejecutar.

In [ ]:
df_raw = pd.read_csv("consumo_energia_sudamerica.csv", sep=";", encoding="latin1")
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
print("=== TIPOS DE DATOS ===")
print(df_raw.dtypes)
print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
display(df_raw.describe().T)

## 2. Calidad de los datos

In [ ]:
plt.figure(figsize=(12, 4))
sns.heatmap(df_raw.isnull(), cbar=False, cmap="Blues", yticklabels=False)
plt.title("Mapa de valores nulos")
plt.tight_layout(); plt.savefig("outputs/01_nulos.png", dpi=120); plt.show()

print("Nulos totales:", int(df_raw.isnull().sum().sum()))
print("Duplicados   :", int(df_raw.duplicated().sum()))

In [ ]:
df = df_raw.copy()
constantes = [c for c in df.columns if df[c].nunique() == 1]
df = df.drop(columns=constantes)
df["panel_solar"] = df["panel_solar"].replace({"SÃ­": "Sí"})

print("Columnas constantes eliminadas (cero información):", constantes)
print("Columnas tras limpieza:", df.shape[1])
print("\nCardinalidad de las categóricas:")
for col in df.select_dtypes(include="object").columns:
    print(f"  {col:24s} {df[col].nunique():3d} valores únicos")

## 3. Análisis exploratorio

### 3.1 Distribuciones de las variables numéricas

In [ ]:
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
n = len(numericas)
filas = int(np.ceil(n / 4))
fig, axes = plt.subplots(filas, 4, figsize=(16, 3 * filas))
for ax, col in zip(axes.ravel(), numericas):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#3b6ea5")
    ax.set_title(col, fontsize=9); ax.set_xlabel("")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.savefig("outputs/02_distribuciones.png", dpi=110); plt.show()

In [ ]:
fig, axes = plt.subplots(filas, 4, figsize=(16, 2.6 * filas))
for ax, col in zip(axes.ravel(), numericas):
    sns.boxplot(x=df[col], ax=ax, color="#7fa8d4")
    ax.set_title(col, fontsize=9); ax.set_xlabel("")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.suptitle("Detección de valores atípicos", y=1.001)
plt.tight_layout(); plt.savefig("outputs/03_outliers.png", dpi=110); plt.show()

### 3.2 Variables categóricas

In [ ]:
categoricas = [c for c in df.select_dtypes(include="object").columns if df[c].nunique() <= 25]
filas_c = int(np.ceil(len(categoricas) / 3))
fig, axes = plt.subplots(filas_c, 3, figsize=(16, 3.4 * filas_c))
for ax, col in zip(axes.ravel(), categoricas):
    orden = df[col].value_counts().index[:12]
    sns.countplot(y=df[col], order=orden, hue=df[col], palette=PALETA, legend=False, ax=ax)
    ax.set_title(col, fontsize=10); ax.set_ylabel("")
for ax in axes.ravel()[len(categoricas):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig("outputs/04_categoricas.png", dpi=110); plt.show()

### 3.3 Matriz de correlación

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="Blues",
            annot_kws={"size": 7}, linewidths=0.3)
plt.title("Matriz de correlación")
plt.tight_layout(); plt.savefig("outputs/05_correlacion.png", dpi=120); plt.show()

### 3.4 El consumo mensual y su contexto geográfico

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["consumo_mes_kwh"], bins=40, kde=True, color="#3b6ea5")
plt.axvline(df["consumo_mes_kwh"].mean(), color="darkorange", linestyle="--",
            label=f"Media: {df['consumo_mes_kwh'].mean():.0f} kWh")
plt.title("Distribución del consumo mensual (kWh)"); plt.legend()
plt.tight_layout(); plt.savefig("outputs/06_consumo.png", dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
prom_pais = df.groupby("país")["consumo_mes_kwh"].mean().sort_values(ascending=False)
sns.barplot(x=prom_pais.values, y=prom_pais.index, hue=prom_pais.index,
            palette=PALETA, legend=False, ax=axes[0])
axes[0].set_title("Consumo promedio por país"); axes[0].set_xlabel("kWh/mes")

top_ciudades = df.groupby("ciudad")["consumo_mes_kwh"].mean().sort_values(ascending=False).head(15)
sns.barplot(x=top_ciudades.values, y=top_ciudades.index, hue=top_ciudades.index,
            palette=PALETA, legend=False, ax=axes[1])
axes[1].set_title("Top 15 ciudades con mayor consumo promedio"); axes[1].set_xlabel("kWh/mes")
plt.tight_layout(); plt.savefig("outputs/07_geografia.png", dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=df, x="estación", y="consumo_mes_kwh", hue="estación",
            palette="Blues", legend=False, ax=axes[0])
axes[0].set_title("Consumo según estación")

sns.boxplot(data=df, x="perfil_uso", y="consumo_mes_kwh", hue="perfil_uso",
            palette="Blues", legend=False, ax=axes[1])
axes[1].set_title("Consumo según perfil de uso"); axes[1].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="horario_mayor_consumo", hue="horario_mayor_consumo",
              palette=PALETA, legend=False, ax=axes[2])
axes[2].set_title("Horario de mayor consumo")
plt.tight_layout(); plt.savefig("outputs/08_contexto.png", dpi=120); plt.show()

**Observación clave:** las diferencias entre países, estaciones y perfiles de uso son mínimas (todas las medias rondan los 410 kWh). Es la primera señal de que el dataset fue **generado sintéticamente** y de que estas variables no aportarán poder predictivo.

### 3.5 Relación de cada factor con el consumo

In [ ]:
muestra = df.sample(3000, random_state=RANDOM_STATE)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.scatterplot(data=muestra, x="personas", y="consumo_mes_kwh", alpha=0.3,
                color="#3b6ea5", ax=axes[0, 0])
axes[0, 0].set_title(f"Personas vs consumo (r = {df['personas'].corr(df['consumo_mes_kwh']):.2f})")

sns.regplot(data=muestra, x="superficie_m2", y="consumo_mes_kwh",
            scatter_kws={"alpha": 0.25, "color": "#3b6ea5"}, line_kws={"color": "darkorange"}, ax=axes[0, 1])
axes[0, 1].set_title(f"Superficie vs consumo (r = {df['superficie_m2'].corr(df['consumo_mes_kwh']):.2f})")

sns.regplot(data=muestra, x="temperatura_promedio", y="consumo_mes_kwh",
            scatter_kws={"alpha": 0.25, "color": "#3b6ea5"}, line_kws={"color": "darkorange"}, ax=axes[1, 0])
axes[1, 0].set_title(f"Temperatura vs consumo (r = {df['temperatura_promedio'].corr(df['consumo_mes_kwh']):.2f})")

sns.regplot(data=muestra, x="porcentaje_led", y="consumo_mes_kwh",
            scatter_kws={"alpha": 0.25, "color": "#3b6ea5"}, line_kws={"color": "darkorange"}, ax=axes[1, 1])
axes[1, 1].set_title(f"Porcentaje LED vs consumo (r = {df['porcentaje_led'].corr(df['consumo_mes_kwh']):.2f})")

plt.tight_layout(); plt.savefig("outputs/09_factores.png", dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=df, x="panel_solar", y="consumo_mes_kwh", hue="panel_solar",
            palette="Blues", legend=False, ax=axes[0])
axes[0].set_title("Paneles solares y consumo")

sns.boxplot(data=df, x="vehiculo_electrico", y="consumo_mes_kwh", hue="vehiculo_electrico",
            palette="Blues", legend=False, ax=axes[1])
axes[1].set_title("Vehículo eléctrico y consumo")

sns.boxplot(data=df, x="medidor_inteligente", y="consumo_mes_kwh", hue="medidor_inteligente",
            palette="Blues", legend=False, ax=axes[2])
axes[2].set_title("Medidor inteligente y consumo")
plt.tight_layout(); plt.savefig("outputs/10_equipamiento.png", dpi=120); plt.show()

print("Consumo medio con/sin panel solar:", df.groupby("panel_solar")["consumo_mes_kwh"].mean().round(1).to_dict())

**Único efecto claro del equipamiento:** los hogares con paneles solares consumen de red unos 62 kWh/mes menos. Por eso `panel_solar` será la palanca dominante del motor de recomendaciones.

## 4. Auditoría de fuga de información

El hallazgo más importante del proyecto, demostrado numérica y visualmente.

**Tres vías de fuga hacia el consumo mensual:**

| Vía | Relación |
|---|---|
| Tramos horarios | `punta + valle + normal = consumo_mes_kwh` (exacto) |
| Financieras | `gasto_mensual_clp = consumo_mes_kwh × valor_kwh_clp` (exacto) |
| Variables derivadas del target | `consumo/personas`, `consumo/superficie`, `gasto/personas` |

Además, la columna `eficiencia_energetica` del dataset **también tiene fuga**: es una función determinista de `porcentaje_led`. Por eso no se usa como variable objetivo.

In [ ]:
suma_tramos = df["consumo_hora_punta_kwh"] + df["consumo_hora_valle_kwh"] + df["consumo_hora_normal_kwh"]
print("Máx |consumo_mes_kwh − suma de tramos| :", (df["consumo_mes_kwh"] - suma_tramos).abs().max())
print("Máx |gasto_mensual_clp − consumo×valor|:", (df["gasto_mensual_clp"] - df["consumo_mes_kwh"] * df["valor_kwh_clp"]).abs().max())

print("\nRango de porcentaje_led por clase de eficiencia_energetica (sin solape = regla determinista):")
display(df.groupby("eficiencia_energetica")["porcentaje_led"].agg(["min", "max", "count"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.scatterplot(data=muestra, x="consumo_mes_kwh", y="gasto_mensual_clp",
                alpha=0.4, color="#c44e52", ax=axes[0])
axes[0].set_title("FUGA VISIBLE: consumo vs gasto (relación perfecta)")

sns.boxplot(data=df, x="eficiencia_energetica", y="porcentaje_led",
            order=["Baja", "Media", "Alta"], hue="eficiencia_energetica",
            palette="Reds", legend=False, ax=axes[1])
axes[1].set_title("FUGA VISIBLE: la etiqueta original sale solo de porcentaje_led")
plt.tight_layout(); plt.savefig("outputs/11_fuga.png", dpi=120); plt.show()

In [ ]:
y_consumo = df["consumo_mes_kwh"]

# (A) CON fuga: replica el planteamiento original
X_fuga = df.drop(columns=["consumo_mes_kwh", "id_vivienda"])
X_fuga["consumo_persona"] = df["consumo_mes_kwh"] / df["personas"]
X_fuga["consumo_m2"] = df["consumo_mes_kwh"] / df["superficie_m2"]
X_fuga["gasto_persona"] = df["gasto_mensual_clp"] / df["personas"]
X_fuga = pd.get_dummies(X_fuga, drop_first=True)
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(X_fuga, y_consumo, test_size=0.2, random_state=RANDOM_STATE)
pred_fuga = LinearRegression().fit(Xf_tr, yf_tr).predict(Xf_te)

# (B) SIN fuga
COLS_FUGA = ["consumo_hora_punta_kwh", "consumo_hora_valle_kwh", "consumo_hora_normal_kwh",
             "gasto_mensual_clp", "valor_kwh_clp", "tarifa_clp_kwh_x100",
             "eficiencia_energetica", "recomendación"]
X_limpio = pd.get_dummies(
    df.drop(columns=["consumo_mes_kwh", "id_vivienda", "ciudad"] + [c for c in COLS_FUGA if c in df.columns]),
    drop_first=True)
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_limpio, y_consumo, test_size=0.2, random_state=RANDOM_STATE)
pred_limpio = LinearRegression().fit(Xl_tr, yl_tr).predict(Xl_te)

display(pd.DataFrame([
    {"Escenario": "CON fuga (planteamiento original)", "R²": r2_score(yf_te, pred_fuga),
     "MAE (kWh)": mean_absolute_error(yf_te, pred_fuga)},
    {"Escenario": "SIN fuga (variables legítimas)", "R²": r2_score(yl_te, pred_limpio),
     "MAE (kWh)": mean_absolute_error(yl_te, pred_limpio)},
]))

**Conclusión:** con las columnas filtradas, una simple Regresión Lineal alcanza **R² = 1.000000 con MAE del orden de 10⁻¹²** — *precisión de máquina*. Ningún fenómeno real se predice con error cero. Al retirarlas, el R² honesto queda en torno a **0.91**.

## 5. ¿Qué campos pedirle a Back-End? Evidencia para ampliar el contrato

Cada campo nuevo es fricción para el usuario del formulario, así que se mide cuánto aporta cada candidato antes de pedirlo.

In [ ]:
candidatos = pd.DataFrame({
    "personas": df["personas"], "superficie_m2": df["superficie_m2"],
    "equipos_electricos": df["equipos_electricos"], "aires_acondicionados": df["aires_acondicionados"],
    "baños": df["baños"], "televisores": df["televisores"], "computadores": df["computadores"],
    "estufas_electricas": df["estufas_electricas"], "temperatura_promedio": df["temperatura_promedio"],
    "vehiculo_electrico": df["vehiculo_electrico"],
    "tipo_vivienda_cod": (df["tipo_vivienda"] == "Departamento").astype(int),
}).astype(float)

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(candidatos, y_consumo, test_size=0.2, random_state=RANDOM_STATE)
seleccionadas, restantes, r2_previo, historial = [], list(candidatos.columns), 0.0, []
while restantes and len(seleccionadas) < 8:
    mejor, mejor_r2 = None, -np.inf
    for cand in restantes:
        cols = seleccionadas + [cand]
        r2 = r2_score(yc_te, LinearRegression().fit(Xc_tr[cols], yc_tr).predict(Xc_te[cols]))
        if r2 > mejor_r2: mejor, mejor_r2 = cand, r2
    historial.append({"Campo añadido": mejor, "R² acumulado": round(mejor_r2, 4),
                       "Aporte": round(mejor_r2 - r2_previo, 4)})
    seleccionadas.append(mejor); restantes.remove(mejor); r2_previo = mejor_r2

hist_df = pd.DataFrame(historial)
display(hist_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(range(1, len(hist_df) + 1), hist_df["R² acumulado"], marker="o", color="#3b6ea5")
axes[0].set_xticks(range(1, len(hist_df) + 1)); axes[0].set_xticklabels(hist_df["Campo añadido"], rotation=45, ha="right")
axes[0].set_title("R² acumulado al añadir campos"); axes[0].set_ylabel("R²")
sns.barplot(data=hist_df, x="Aporte", y="Campo añadido", hue="Campo añadido",
            palette=PALETA, legend=False, ax=axes[1])
axes[1].set_title("Aporte marginal de cada campo")
plt.tight_layout(); plt.savefig("outputs/12_seleccion.png", dpi=120); plt.show()

**`personas` y `superficie_m2` aportan todo el poder predictivo disponible (R² de 0 → 0.88). El resto aporta exactamente cero.** Por eso el contrato se amplía con dos campos, no con diez.

> **Matiz honesto:** que el resto aporte cero es casi con certeza un artefacto de la generación sintética. En datos reales, el aire acondicionado y la temperatura sí influyen físicamente. El contrato debe diseñarse extensible.

## 6. Por qué `personas` y `superficie_m2` son OBLIGATORIOS

Se evaluó permitir que faltaran e imputar la mediana. **Se descartó con datos.**

In [ ]:
NORMALIZADORES = ["personas", "superficie_m2", "cantidad_equipos", "tipo_inmueble_cod"]
base = pd.DataFrame({
    "personas": df["personas"].astype(float), "superficie_m2": df["superficie_m2"].astype(float),
    "cantidad_equipos": df["equipos_electricos"].astype(float),
    "tipo_inmueble_cod": (df["tipo_vivienda"] == "Departamento").astype(float)})
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(base, y_consumo, test_size=0.2, random_state=RANDOM_STATE)
modelo_prueba = LinearRegression().fit(Xn_tr, yn_tr)

residuo_full = y_consumo - modelo_prueba.predict(base)
c40, c75 = residuo_full.quantile([0.40, 0.75])
cat_prueba = lambda r: "Eficiente" if r <= c40 else ("Moderado" if r <= c75 else "Ineficiente")
cat_full = residuo_full.apply(cat_prueba)

base_imp = base.copy()
base_imp["personas"] = df["personas"].median()
base_imp["superficie_m2"] = df["superficie_m2"].median()
cat_red = (y_consumo - modelo_prueba.predict(base_imp)).apply(cat_prueba)

coincidencia = (cat_full == cat_red).mean()
azar = (cat_full.value_counts(normalize=True) ** 2).sum()
graves = ((cat_full == "Eficiente") & (cat_red == "Ineficiente")).sum() + \
         ((cat_full == "Ineficiente") & (cat_red == "Eficiente")).sum()
print(f"Coincidencia completo vs imputado : {coincidencia*100:.1f}%   (azar: {azar*100:.1f}%)")
print(f"Evaluaciones invertidas           : {graves} ({graves/len(df)*100:.1f}%)")

plt.figure(figsize=(5.5, 4))
sns.heatmap(pd.crosstab(cat_full, cat_red), annot=True, fmt="d", cmap="Reds")
plt.title("Modo completo vs. imputado: la diagonal debería dominar")
plt.xlabel("Categoría con imputación"); plt.ylabel("Categoría correcta")
plt.tight_layout(); plt.savefig("outputs/13_imputacion.png", dpi=120); plt.show()

**La degradación elegante no es viable.** La coincidencia (≈42%) apenas supera el azar (≈35%) y casi uno de cada cinco hogares recibe la evaluación *invertida*. `personas` y `superficie_m2` **son** el 88% de la señal: imputarlas no degrada la medición, la destruye.

**Decisión:** ambos campos son obligatorios. Si faltan, la API responde `400`.

## 7. Roles de las variables

| Rol | Variables | Uso |
|---|---|---|
| **Target** | `consumo_mes_kwh` | Lo que predice el Modelo A |
| **Normalizadores** | `personas`, `superficie_m2`, `cantidad_equipos`, `tipo_inmueble` | Entradas del Modelo A → consumo esperado |
| **Palancas** | `uso_horario_pico`, `horas_alto_consumo`, `panel_solar` | Explican el residuo → recomendaciones |

Las palancas **nunca** entran al Modelo A: contaminarían la definición de "consumo esperado" con el comportamiento que se quiere evaluar.

In [ ]:
d = pd.DataFrame(index=df.index)
d["consumo_kwh"] = df["consumo_mes_kwh"].astype(float)
d["personas"] = df["personas"].astype(float)
d["superficie_m2"] = df["superficie_m2"].astype(float)
d["cantidad_equipos"] = df["equipos_electricos"].astype(float)
d["tipo_inmueble_cod"] = (df["tipo_vivienda"] == "Departamento").astype(float)
tramos = df[["consumo_hora_punta_kwh", "consumo_hora_valle_kwh", "consumo_hora_normal_kwh"]]
d["uso_horario_pico"] = (tramos.idxmax(axis=1) == "consumo_hora_punta_kwh").astype(float)
d["horas_alto_consumo"] = np.clip(df["consumo_hora_punta_kwh"] / df["consumo_mes_kwh"] * 24, 0, 16).round(1)
d["panel_solar"] = (df["panel_solar"] == "Sí").astype(float)

PALANCAS = ["uso_horario_pico", "horas_alto_consumo", "panel_solar"]
print("Normalizadores:", NORMALIZADORES)
print("Palancas      :", PALANCAS)
d.head(3)

## 8. Modelo A — Consumo esperado

Se compara Regresión Lineal y Random Forest, siempre contra un baseline trivial.

In [ ]:
X_A = d[NORMALIZADORES].astype(np.float32)
y_A = d["consumo_kwh"]
XA_tr, XA_te, yA_tr, yA_te = train_test_split(X_A, y_A, test_size=0.2, random_state=RANDOM_STATE)

filas_A = []
for nombre, modelo in [("Regresión Lineal", LinearRegression()),
                        ("Random Forest", RandomForestRegressor(n_estimators=300, max_depth=10,
                                                                 random_state=RANDOM_STATE, n_jobs=-1))]:
    modelo.fit(XA_tr, yA_tr)
    pred = modelo.predict(XA_te)
    cv = cross_val_score(modelo, X_A, y_A, cv=5, scoring="r2")
    filas_A.append({"Modelo": nombre, "R²": round(r2_score(yA_te, pred), 4),
                     "MAE (kWh)": round(mean_absolute_error(yA_te, pred), 2),
                     "R² CV": round(cv.mean(), 4), "CV std": round(cv.std(), 4)})
pred_base = np.full_like(yA_te, yA_tr.mean(), dtype=float)
filas_A.append({"Modelo": "Baseline (media)", "R²": round(r2_score(yA_te, pred_base), 4),
                 "MAE (kWh)": round(mean_absolute_error(yA_te, pred_base), 2), "R² CV": None, "CV std": None})
tabla_A = pd.DataFrame(filas_A)
display(tabla_A)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=tabla_A, x="R²", y="Modelo", hue="Modelo", palette=PALETA, legend=False, ax=axes[0])
axes[0].set_title("Comparación de modelos — R²")
sns.barplot(data=tabla_A, x="MAE (kWh)", y="Modelo", hue="Modelo", palette=PALETA, legend=False, ax=axes[1])
axes[1].set_title("Comparación de modelos — MAE")
plt.tight_layout(); plt.savefig("outputs/14_modelos.png", dpi=120); plt.show()

**Se elige Regresión Lineal:** mejor R², **extrapola correctamente** fuera del rango de entrenamiento (los árboles se quedan pegados a la hoja más cercana, problemático para una API pública) y produce un grafo ONNX mínimo.

In [ ]:
modelo_A = LinearRegression().fit(XA_tr, yA_tr)
d["consumo_esperado"] = modelo_A.predict(d[NORMALIZADORES].astype(np.float32))
d["residuo"] = d["consumo_kwh"] - d["consumo_esperado"]

coef_A = pd.DataFrame({"Variable": NORMALIZADORES, "Coeficiente (kWh/mes)": modelo_A.coef_})
print(f"Intercepto: {modelo_A.intercept_:.3f} kWh/mes")
display(coef_A.round(3))

plt.figure(figsize=(7, 3))
sns.barplot(data=coef_A, x="Coeficiente (kWh/mes)", y="Variable", hue="Variable",
            palette=PALETA, legend=False)
plt.axvline(0, color="black", linewidth=1)
plt.title("Modelo A — efecto de cada normalizador sobre el consumo")
plt.tight_layout(); plt.savefig("outputs/15_coeficientes.png", dpi=120); plt.show()

Los coeficientes son **físicamente interpretables**: cada persona adicional suma ~35 kWh/mes y cada m² suma ~1.3 kWh/mes. Ese realismo indica que el modelo aprende estructura real y no ruido.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
muestra_d = d.sample(2500, random_state=RANDOM_STATE)

sns.scatterplot(data=muestra_d, x="consumo_esperado", y="consumo_kwh", alpha=0.3,
                color="#3b6ea5", ax=axes[0])
lims = [d["consumo_kwh"].min(), d["consumo_kwh"].max()]
axes[0].plot(lims, lims, "--", color="black", linewidth=1)
axes[0].set_title("Consumo real vs. esperado")

sns.histplot(d["residuo"], bins=50, kde=True, color="#3b6ea5", ax=axes[1])
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Distribución del residuo"); axes[1].set_xlabel("kWh")

sns.scatterplot(data=muestra_d, x="consumo_esperado", y="residuo", alpha=0.3,
                color="#3b6ea5", ax=axes[2])
axes[2].axhline(0, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Residuos vs. predicción (deben ser aleatorios)")
plt.tight_layout(); plt.savefig("outputs/16_diagnostico.png", dpi=120); plt.show()

## 9. Categorización por percentiles del residuo

In [ ]:
CORTE_EFICIENTE = float(d["residuo"].quantile(0.40))
CORTE_MODERADO = float(d["residuo"].quantile(0.75))

def categorizar(residuo: float) -> str:
    if residuo <= CORTE_EFICIENTE: return "Eficiente"
    elif residuo <= CORTE_MODERADO: return "Moderado"
    return "Ineficiente"

d["categoria"] = d["residuo"].apply(categorizar)
print(f"Eficiente  : residuo <= {CORTE_EFICIENTE:.3f} kWh")
print(f"Moderado   : residuo <= {CORTE_MODERADO:.3f} kWh")
print(f"Ineficiente: residuo >  {CORTE_MODERADO:.3f} kWh")
print("\nDistribución:", d["categoria"].value_counts(normalize=True).round(3).to_dict())

orden_cat = ["Eficiente", "Moderado", "Ineficiente"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x=d["categoria"], order=orden_cat, hue=d["categoria"], palette="Blues", legend=False, ax=axes[0])
axes[0].set_title("Distribución de categorías"); axes[0].set_xlabel("")
sns.boxplot(data=d, x="categoria", y="consumo_kwh", order=orden_cat, hue="categoria",
            palette="Blues", legend=False, ax=axes[1])
axes[1].set_title("Consumo real por categoría"); axes[1].set_xlabel("")
sns.boxplot(data=d, x="categoria", y="residuo", order=orden_cat, hue="categoria",
            palette="Blues", legend=False, ax=axes[2])
axes[2].axhline(0, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Residuo por categoría"); axes[2].set_xlabel("")
plt.tight_layout(); plt.savefig("outputs/17_categorias.png", dpi=120); plt.show()

## 10. Modelo B — Atribución del residuo a las palancas

Modelo **interpretable a propósito**: los coeficientes *son* el producto, porque de ellos nacen las recomendaciones.

In [ ]:
X_B, y_B = d[PALANCAS], d["residuo"]
XB_tr, XB_te, yB_tr, yB_te = train_test_split(X_B, y_B, test_size=0.2, random_state=RANDOM_STATE)
escalador_B = StandardScaler().fit(XB_tr)
modelo_B = Ridge(alpha=1.0).fit(escalador_B.transform(XB_tr), yB_tr)
pred_B = modelo_B.predict(escalador_B.transform(XB_te))

print(f"Modelo B -> R²: {r2_score(yB_te, pred_B):.4f} | MAE: {mean_absolute_error(yB_te, pred_B):.2f} kWh")
coef_B = pd.DataFrame({"Palanca": PALANCAS, "Coeficiente": modelo_B.coef_}).sort_values("Coeficiente")
display(coef_B.round(3))

plt.figure(figsize=(7, 3))
sns.barplot(data=coef_B, x="Coeficiente", y="Palanca", hue="Palanca", palette=PALETA, legend=False)
plt.axvline(0, color="black", linewidth=1)
plt.title("Modelo B — efecto de cada palanca sobre el residuo (negativo = mejora)")
plt.tight_layout(); plt.savefig("outputs/18_atribucion.png", dpi=120); plt.show()

**Lectura honesta:** R² ≈ 0.23. Las palancas explican una fracción real pero modesta del residuo, y `panel_solar` domina de forma abrumadora frente a las conductuales. Es un hallazgo del dataset, no un defecto de implementación.

## 11. Motor de recomendaciones (coherente por construcción)

**Regla estructural:** si la categoría es `Eficiente`, solo refuerzo positivo. Un hogar que consume menos de lo esperado no puede recibir una recomendación de reducir.

In [ ]:
REFERENCIA_BUENA = {"uso_horario_pico": 0.0,
                     "horas_alto_consumo": float(d["horas_alto_consumo"].quantile(0.25)),
                     "panel_solar": 1.0}
TEXTOS = {"uso_horario_pico": "Reducir el uso de equipos durante los horarios pico",
          "horas_alto_consumo": "Distribuir las actividades de mayor consumo a lo largo del día",
          "panel_solar": "Evaluar la instalación de paneles solares"}

def generar_recomendaciones(palancas: dict, categoria: str) -> list:
    if categoria == "Eficiente":
        return ["Mantener los hábitos actuales: consumes menos de lo esperado para un hogar como el tuyo"]
    actual = pd.DataFrame([[palancas[p] for p in PALANCAS]], columns=PALANCAS)
    referencia = pd.DataFrame([[REFERENCIA_BUENA[p] for p in PALANCAS]], columns=PALANCAS)
    contribucion = -modelo_B.coef_ * (escalador_B.transform(actual)[0] - escalador_B.transform(referencia)[0])
    orden = np.argsort(-contribucion)
    recs = [TEXTOS[PALANCAS[i]] for i in orden if contribucion[i] > 0.01][:3]
    return recs or ["Revisar hábitos generales de consumo"]

## 12. Función de análisis integral

In [ ]:
TARIFA_REFERENCIA_USD_KWH = 0.75

def codificar_normalizadores(entrada: dict) -> np.ndarray:
    return np.array([[float(entrada["personas"]), float(entrada["superficie_m2"]),
                      float(entrada["cantidad_equipos"]),
                      1.0 if entrada["tipo_inmueble"] == "Departamento" else 0.0]], dtype=np.float32)

def analizar_perfil_energetico(entrada: dict) -> dict:
    for obligatorio in ["consumo_kwh", "personas", "superficie_m2", "cantidad_equipos", "tipo_inmueble"]:
        if entrada.get(obligatorio) is None:
            raise ValueError(f"Campo obligatorio ausente: {obligatorio}")

    consumo_esperado = float(modelo_A.predict(codificar_normalizadores(entrada))[0])
    residuo = float(entrada["consumo_kwh"]) - consumo_esperado
    categoria = categorizar(residuo)
    palancas = {"uso_horario_pico": float(bool(entrada.get("uso_horario_pico", False))),
                "horas_alto_consumo": float(entrada.get("horas_alto_consumo", 0)),
                "panel_solar": float(bool(entrada.get("panel_solar", False)))}
    desviacion = (residuo / consumo_esperado * 100) if consumo_esperado > 0 else 0.0

    return {"categoria": categoria,
            "consumo_esperado_kwh": round(consumo_esperado, 1),
            "residuo_kwh": round(residuo, 1),
            "desviacion_porcentual": round(desviacion, 1),
            "recomendaciones": generar_recomendaciones(palancas, categoria),
            "costo_estimado_mensual": round(float(entrada["consumo_kwh"]) * TARIFA_REFERENCIA_USD_KWH, 2)}

## 13. Test de coherencia automatizado (quality gate)

**Si esta celda falla, el sistema no debe entregarse.**

In [ ]:
rng_test = np.random.default_rng(7)
N_PRUEBA = 5000
casos = pd.DataFrame({
    "consumo_kwh": rng_test.uniform(50, 900, N_PRUEBA).round(1),
    "cantidad_equipos": rng_test.integers(1, 25, N_PRUEBA),
    "tipo_inmueble": rng_test.choice(["Casa", "Departamento"], N_PRUEBA),
    "uso_horario_pico": rng_test.choice([True, False], N_PRUEBA),
    "horas_alto_consumo": rng_test.uniform(0, 16, N_PRUEBA).round(1),
    "personas": rng_test.integers(1, 8, N_PRUEBA),
    "superficie_m2": rng_test.uniform(30, 300, N_PRUEBA).round(0),
    "panel_solar": rng_test.choice([True, False], N_PRUEBA)})
salidas = [analizar_perfil_energetico(f.to_dict()) for _, f in casos.iterrows()]

efic_con_reduccion = sum(1 for s in salidas
                          if s["categoria"] == "Eficiente" and "Mantener" not in s["recomendaciones"][0])
noefic_sin_recs = sum(1 for s in salidas
                      if s["categoria"] != "Eficiente" and "Mantener" in s["recomendaciones"][0])
signo_incorrecto = sum(1 for s in salidas
                       if (s["categoria"] == "Eficiente") != (s["residuo_kwh"] <= CORTE_EFICIENTE))

print(f"Casos evaluados: {N_PRUEBA}")
print(f"  'Eficiente' con recomendación de reducir  : {efic_con_reduccion} (debe ser 0)")
print(f"  No-'Eficiente' sin recomendación de mejora: {noefic_sin_recs} (debe ser 0)")
print(f"  Categoría incoherente con el residuo      : {signo_incorrecto} (debe ser 0)")
assert efic_con_reduccion == 0 and noefic_sin_recs == 0 and signo_incorrecto == 0, "TEST DE COHERENCIA FALLIDO"
print("\nTest de coherencia superado.")

resultados_prueba = pd.DataFrame({"categoria": [s["categoria"] for s in salidas],
                                   "residuo": [s["residuo_kwh"] for s in salidas]})
plt.figure(figsize=(6, 4))
sns.boxplot(data=resultados_prueba, x="categoria", y="residuo", order=orden_cat,
            hue="categoria", palette="Blues", legend=False)
plt.axhline(CORTE_EFICIENTE, color="green", linestyle="--", linewidth=1, label="corte Eficiente")
plt.axhline(CORTE_MODERADO, color="darkorange", linestyle="--", linewidth=1, label="corte Moderado")
plt.title(f"Coherencia sobre {N_PRUEBA} casos sintéticos"); plt.legend(); plt.xlabel("")
plt.tight_layout(); plt.savefig("outputs/19_coherencia.png", dpi=120); plt.show()

## 14. Ejemplos de uso

El primer ejemplo es el caso canónico del enunciado. **Cambia de categoría** respecto a la versión de 5 campos: para un hogar de 4 personas y 120 m², 420 kWh está *por debajo* de lo esperado. Ese es el valor que añade el contrato ampliado.

In [ ]:
ejemplos = [
    {"consumo_kwh": 420, "uso_horario_pico": True, "cantidad_equipos": 10, "tipo_inmueble": "Casa",
     "horas_alto_consumo": 8, "personas": 4, "superficie_m2": 120, "panel_solar": False},
    {"consumo_kwh": 210, "uso_horario_pico": False, "cantidad_equipos": 6, "tipo_inmueble": "Departamento",
     "horas_alto_consumo": 3, "personas": 2, "superficie_m2": 55, "panel_solar": True},
    {"consumo_kwh": 620, "uso_horario_pico": True, "cantidad_equipos": 18, "tipo_inmueble": "Casa",
     "horas_alto_consumo": 11, "personas": 3, "superficie_m2": 90, "panel_solar": False}]
salidas_ejemplos = [analizar_perfil_energetico(e) for e in ejemplos]
for e, s in zip(ejemplos, salidas_ejemplos):
    print("Entrada:", json.dumps(e, ensure_ascii=False))
    print("Salida :", json.dumps(s, ensure_ascii=False, indent=2))
    print("-" * 72)
with open("outputs/v4_ejemplos.json", "w", encoding="utf-8") as f:
    json.dump([{"entrada": e, "salida": s} for e, s in zip(ejemplos, salidas_ejemplos)], f,
               ensure_ascii=False, indent=2)

## 15. Serialización

Solo el **Modelo A** se exporta a ONNX. El Modelo B son tres coeficientes: entregarlos como constantes en el JSON es más simple y menos frágil que un segundo archivo.

In [ ]:
joblib.dump(modelo_A, "models/v4_modelo_A_consumo_esperado.pkl")
joblib.dump({"modelo": modelo_B, "escalador": escalador_B}, "models/v4_modelo_B_atribucion.pkl")
print("Modelos guardados en formato joblib.")

In [ ]:
!pip install -q skl2onnx onnx

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

onnx_A = convert_sklearn(modelo_A,
                          initial_types=[("input", FloatTensorType([None, len(NORMALIZADORES)]))],
                          target_opset=15)
with open("models/v4_modelo_A.onnx", "wb") as f:
    f.write(onnx_A.SerializeToString())
print("Guardado: models/v4_modelo_A.onnx")

In [ ]:
import onnxruntime as ort

sesion = ort.InferenceSession("models/v4_modelo_A.onnx")
nombre_entrada = sesion.get_inputs()[0].name
print("Entrada:", nombre_entrada, "| forma:", sesion.get_inputs()[0].shape)
print("Salidas:", [o.name for o in sesion.get_outputs()])

muestra_onnx = XA_te.values[:1000].astype(np.float32)
pred_onnx = np.array(sesion.run(None, {nombre_entrada: muestra_onnx})[0]).ravel()
dif_max = np.abs(pred_onnx - modelo_A.predict(muestra_onnx)).max()
print(f"\nDiferencia máxima ONNX vs sklearn: {dif_max:.2e} kWh")
assert dif_max < 1e-3, "VERIFICACIÓN ONNX FALLIDA"
print("Verificación ONNX superada.")

## 16. Esquema de entrega para Back-End

In [ ]:
esquema = {
    "version": "v4 - contrato ampliado (residuo)",
    "modelo_A": {"archivo_onnx": "v4_modelo_A.onnx",
                  "orden_features": NORMALIZADORES,
                  "coeficientes": {v: round(float(c), 6) for v, c in zip(NORMALIZADORES, modelo_A.coef_)},
                  "intercepto": round(float(modelo_A.intercept_), 6),
                  "metricas": {"R2": round(r2_score(yA_te, modelo_A.predict(XA_te)), 4),
                                "MAE_kwh": round(mean_absolute_error(yA_te, modelo_A.predict(XA_te)), 2)}},
    "cortes_categoria_kwh": {"eficiente_max": round(CORTE_EFICIENTE, 4),
                              "moderado_max": round(CORTE_MODERADO, 4)},
    "modelo_B_para_java": {"orden_palancas": PALANCAS,
                            "coeficientes": {p: round(float(c), 6) for p, c in zip(PALANCAS, modelo_B.coef_)},
                            "escalador_media": {p: round(float(m), 6) for p, m in zip(PALANCAS, escalador_B.mean_)},
                            "escalador_desviacion": {p: round(float(s), 6) for p, s in zip(PALANCAS, escalador_B.scale_)},
                            "referencia_buena": {k: round(float(v), 6) for k, v in REFERENCIA_BUENA.items()}},
    "campos_obligatorios": ["consumo_kwh", "personas", "superficie_m2", "cantidad_equipos", "tipo_inmueble"],
    "campos_opcionales": ["uso_horario_pico", "horas_alto_consumo", "panel_solar"],
    "codificacion": {"tipo_inmueble": {"Casa": 0.0, "Departamento": 1.0}},
    "tarifa_usd_kwh": TARIFA_REFERENCIA_USD_KWH,
    "test_coherencia": f"Superado sobre {N_PRUEBA} combinaciones (0 incoherencias)",
    "nota_obligatoriedad": "personas y superficie_m2 son OBLIGATORIOS. Imputarlos produce 19.6% de evaluaciones invertidas. Si faltan, responder 400."}
with open("models/v4_esquema.json", "w", encoding="utf-8") as f:
    json.dump(esquema, f, ensure_ascii=False, indent=2)
print(json.dumps(esquema, ensure_ascii=False, indent=2))

## 17. Resumen

| Aspecto | Versión de 5 campos | **v4 (contrato ampliado)** |
|---|---|---|
| Definición de eficiencia | Índice de intensidad (circular) | **Residuo vs. hogares similares (no circular)** |
| Target del modelo | Etiqueta construida | **Consumo real medido** |
| Validación | F1 0.83 vs baseline 0.19 | **R² 0.88 vs baseline ≈0** |
| Mensaje al usuario | Percentil de intensidad | **"Consumes X% más/menos que hogares como el tuyo"** |
| Interpretabilidad | Importancia de variables | **+35 kWh por persona, +1.3 kWh por m²** |

**Gráficos generados** en `outputs/`: 19 figuras cubriendo calidad de datos, distribuciones, contexto geográfico, factores de consumo, evidencia visual de la fuga, selección de campos, diagnóstico del modelo, atribución y coherencia.

**Limitaciones declaradas:** el dataset presenta indicios claros de generación sintética (identidades exactas entre columnas, cero nulos en 30.000 filas, diferencias mínimas entre países y estaciones). Solo `personas` y `superficie_m2` muestran relación real con el consumo, y el Modelo B explica apenas el 23% del residuo. La metodología es transferible a datos reales de medidor sin cambios estructurales.